# Computer Vision with PyTorch

Welcome to the **Computer Vision** notebook! Building on the deep-learning fundamentals you've already learned, we now turn to one of the most exciting application areas of deep learning — teaching machines to *see*.

## What is Computer Vision?

Computer vision is the field of AI that enables computers to extract meaningful information from images and video. Practical applications include:

| Domain | Examples |
|--------|----------|
| Healthcare | Medical image analysis, X-ray classification |
| Autonomous vehicles | Object detection, lane tracking |
| Manufacturing | Defect inspection, quality control |
| Security | Face recognition, anomaly detection |
| Agriculture | Crop disease detection, yield estimation |

## How CNNs Work for Images

**Convolutional Neural Networks (CNNs)** are the workhorse of modern computer vision. Unlike fully-connected networks that treat every pixel independently, CNNs exploit the *spatial structure* of images through:

1. **Local connectivity** — each neuron looks at a small patch (receptive field) rather than the entire image.
2. **Weight sharing** — the same filter is applied across the whole image, drastically reducing parameters.
3. **Hierarchical features** — early layers detect edges and textures; deeper layers recognise parts and objects.

In this notebook we will build a CNN from scratch, train it on synthetic data, and evaluate its performance — all without downloading any external datasets.

## Setup

Let's import the libraries we'll use throughout this notebook. We rely only on **NumPy**, **PyTorch**, **torchvision.transforms**, and **Matplotlib**.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

---
# Part 1 — Image Fundamentals

## Images as Tensors

A digital image is a grid of **pixels**. Each pixel holds one or more numeric values:

| Image type | Shape (H × W × C) | Channels |
|---|---|---|
| Grayscale | H × W × 1 | Intensity |
| RGB colour | H × W × 3 | Red, Green, Blue |

PyTorch follows the **channels-first** convention: `(C, H, W)`.

Pixel values typically range from **0** (black) to **255** (white) for 8-bit images, or **0.0–1.0** when normalised as floats.

### Creating Synthetic Images with NumPy

Let's create a few simple images to understand their tensor representation.

In [ ]:
# Create a small 8x8 grayscale image — a simple gradient
gradient = np.linspace(0, 1, 64).reshape(8, 8).astype(np.float32)

# Create a checkerboard pattern
checker = np.zeros((8, 8), dtype=np.float32)
checker[::2, ::2] = 1.0
checker[1::2, 1::2] = 1.0

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(gradient, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Gradient (8×8)")
axes[1].imshow(checker, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Checkerboard (8×8)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Gradient shape: {gradient.shape}, dtype: {gradient.dtype}")
print(f"Pixel value range: [{gradient.min():.2f}, {gradient.max():.2f}]")

### RGB Images

An RGB image has three channels. Let's create one and inspect its structure.

In [ ]:
# Create a 64x64 RGB image with distinct colour regions
rgb_image = np.zeros((64, 64, 3), dtype=np.float32)
rgb_image[:, :21, 0] = 1.0        # Red stripe
rgb_image[:, 21:42, 1] = 1.0      # Green stripe
rgb_image[:, 42:, 2] = 1.0        # Blue stripe

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].imshow(rgb_image)
axes[0].set_title("RGB Combined")
channel_names = ["Red", "Green", "Blue"]
cmaps = ["Reds", "Greens", "Blues"]
for i in range(3):
    axes[i + 1].imshow(rgb_image[:, :, i], cmap=cmaps[i], vmin=0, vmax=1)
    axes[i + 1].set_title(f"{channel_names[i]} Channel")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Convert to PyTorch channels-first format
tensor_image = torch.from_numpy(rgb_image).permute(2, 0, 1)  # HWC -> CHW
print(f"NumPy shape (HWC): {rgb_image.shape}")
print(f"Torch shape (CHW): {tensor_image.shape}")

---
# Part 2 — Image Transformations

Data augmentation and preprocessing are critical in computer vision. `torchvision.transforms` provides a rich set of operations. Let's explore the most common ones on a synthetic image.

In [ ]:
# Create a more interesting synthetic image — a white circle on dark background
def make_circle_image(size=64):
    img = np.zeros((size, size, 3), dtype=np.float32)
    cx, cy, r = size // 2, size // 2, size // 4
    yy, xx = np.ogrid[:size, :size]
    mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
    img[mask] = [0.9, 0.3, 0.1]  # orange circle
    img[~mask] = [0.1, 0.1, 0.3]  # dark blue background
    return img

original = make_circle_image(64)
plt.figure(figsize=(2, 2))
plt.imshow(original)
plt.title("Original")
plt.axis("off")
plt.show()

### Common Transforms

We will apply several transforms and compare the results side by side.

In [ ]:
from PIL import Image

# Convert numpy to PIL for torchvision transforms
pil_img = Image.fromarray((original * 255).astype(np.uint8))

# Define transforms
transform_list = {
    "Original": transforms.ToTensor(),
    "Resize 32×32": transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor()]),
    "H-Flip": transforms.Compose([transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor()]),
    "V-Flip": transforms.Compose([transforms.RandomVerticalFlip(p=1.0), transforms.ToTensor()]),
    "Rotation 45°": transforms.Compose([transforms.RandomRotation((45, 45)), transforms.ToTensor()]),
    "Color Jitter": transforms.Compose([
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.3),
        transforms.ToTensor(),
    ]),
}

fig, axes = plt.subplots(1, len(transform_list), figsize=(18, 3))
for ax, (name, t) in zip(axes, transform_list.items()):
    torch.manual_seed(42)
    tensor = t(pil_img)
    ax.imshow(tensor.permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.suptitle("Common Image Transforms", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Normalization

Normalization shifts pixel values to have zero mean and unit variance. This helps networks train faster and more stably.

```
Normalize(mean, std)  →  output = (input - mean) / std
```

In [ ]:
to_tensor = transforms.ToTensor()
tensor_img = to_tensor(pil_img)

# Compute per-channel stats before normalization
print("Before normalization:")
for i, ch in enumerate(["R", "G", "B"]):
    print(f"  {ch}: mean={tensor_img[i].mean():.4f}, std={tensor_img[i].std():.4f}")

# Normalize with ImageNet-style stats (commonly used)
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225])
normed = normalize(tensor_img)

print("\nAfter normalization:")
for i, ch in enumerate(["R", "G", "B"]):
    print(f"  {ch}: mean={normed[i].mean():.4f}, std={normed[i].std():.4f}")

---
# Part 3 — Convolutional Operations

A **convolution** slides a small kernel (filter) over the image, computing element-wise multiplications and summing the results at each position. Different kernels detect different features.

### Convolution Kernels

Let's visualize three classic kernels and their effects on an image.

In [ ]:
# Create a grayscale test image with sharp edges
test_img = np.zeros((64, 64), dtype=np.float32)
test_img[16:48, 16:48] = 1.0  # white square on black background
# Add a diagonal line
for i in range(64):
    if 0 <= i < 64:
        test_img[i, min(i, 63)] = 1.0

# Define kernels
kernels = {
    "Edge Detection\n(Sobel-X)": np.array([[-1, 0, 1],
                                             [-2, 0, 2],
                                             [-1, 0, 1]], dtype=np.float32),
    "Blur\n(Box 3x3)": np.ones((3, 3), dtype=np.float32) / 9.0,
    "Sharpen": np.array([[ 0, -1,  0],
                          [-1,  5, -1],
                          [ 0, -1,  0]], dtype=np.float32),
}

# Apply convolutions using PyTorch
input_tensor = torch.from_numpy(test_img).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(test_img, cmap="gray")
axes[0].set_title("Input Image")
axes[0].axis("off")

for idx, (name, kernel) in enumerate(kernels.items()):
    k = torch.from_numpy(kernel).unsqueeze(0).unsqueeze(0)  # (1,1,3,3)
    output = F.conv2d(input_tensor, k, padding=1)
    axes[idx + 1].imshow(output.squeeze().numpy(), cmap="gray")
    axes[idx + 1].set_title(name)
    axes[idx + 1].axis("off")

plt.suptitle("Effect of Different Convolution Kernels", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Pooling Operations

**Pooling** reduces the spatial dimensions, making the network more efficient and somewhat invariant to small translations.

- **Max Pooling** — takes the maximum value in each patch (preserves strong activations).
- **Average Pooling** — takes the mean (smoother, less aggressive).

In [ ]:
# Create a feature-map-like image
feature_map = np.random.RandomState(42).rand(1, 1, 8, 8).astype(np.float32)
fm_tensor = torch.from_numpy(feature_map)

max_pool = F.max_pool2d(fm_tensor, kernel_size=2, stride=2)
avg_pool = F.avg_pool2d(fm_tensor, kernel_size=2, stride=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
data = [
    (feature_map.squeeze(), "Original (8×8)"),
    (max_pool.squeeze().numpy(), "Max Pool 2×2 → (4×4)"),
    (avg_pool.squeeze().numpy(), "Avg Pool 2×2 → (4×4)"),
]
for ax, (img, title) in zip(axes, data):
    im = ax.imshow(img, cmap="viridis", vmin=0, vmax=1)
    ax.set_title(title)
    # Annotate cell values
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            ax.text(j, i, f"{img[i, j]:.2f}", ha="center", va="center",
                    fontsize=7, color="white" if img[i, j] < 0.5 else "black")
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

print(f"Original size:   {feature_map.squeeze().shape}")
print(f"After 2×2 pool:  {max_pool.squeeze().shape}")

### Feature Maps Through a Convolution Layer

Let's pass our test image through a Conv2d layer and visualize the learned feature maps.

In [ ]:
torch.manual_seed(42)

conv_layer = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, padding=1)
with torch.no_grad():
    feature_maps = conv_layer(input_tensor)  # input_tensor from earlier (1,1,64,64)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes[0, 0].imshow(test_img, cmap="gray")
axes[0, 0].set_title("Input")
axes[0, 0].axis("off")
axes[0, 1].axis("off")  # empty cell

for i in range(6):
    row, col = divmod(i, 4) if i < 4 else (1, i - 4) if i < 8 else (1, i - 4)
    if i < 2:
        ax = axes[0, i + 2]
    else:
        ax = axes[1, i - 2]
    ax.imshow(feature_maps[0, i].detach().numpy(), cmap="viridis")
    ax.set_title(f"Filter {i}")
    ax.axis("off")

plt.suptitle("Feature Maps from Conv2d (6 filters, 3×3)", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Input shape:       {input_tensor.shape}")
print(f"Feature map shape: {feature_maps.shape}")

---
# Part 4 — Building a CNN Architecture

Now we'll build a complete CNN for image classification using `nn.Module`. The architecture follows a classic pattern:

```
Input → [Conv → ReLU → MaxPool] × 2 → Flatten → Linear → Output
```

We design it for **28×28 grayscale** images with **3 classes** (the shapes dataset we'll create next).

In [ ]:
class ShapeCNN(nn.Module):
    """CNN for classifying 28x28 grayscale images into 3 shape classes."""

    def __init__(self, num_classes=3):
        super().__init__()
        # Block 1: Conv → ReLU → MaxPool
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)   # -> (16, 28, 28)
        self.pool1 = nn.MaxPool2d(2, 2)                            # -> (16, 14, 14)

        # Block 2: Conv → ReLU → MaxPool
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # -> (32, 14, 14)
        self.pool2 = nn.MaxPool2d(2, 2)                            # -> (32, 7, 7)

        # Classifier head
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = ShapeCNN(num_classes=3)
print(model)

### Model Summary and Parameter Count

Let's verify the tensor shapes through the network and count the trainable parameters.

In [ ]:
# Trace shapes through the network
dummy = torch.randn(1, 1, 28, 28)
print("Shape trace:")
x = dummy
for name, layer in [("conv1", model.conv1), ("pool1", model.pool1),
                     ("conv2", model.conv2), ("pool2", model.pool2),
                     ("flatten", model.flatten), ("fc1", model.fc1),
                     ("fc2", model.fc2)]:
    if name in ("conv1", "conv2"):
        x = F.relu(layer(x))
    elif name == "fc1":
        x = F.relu(layer(x))
    else:
        x = layer(x)
    print(f"  {name:>8s} → {str(list(x.shape)):>20s}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

---
# Part 5 — Dataset Preparation

Instead of downloading MNIST or CIFAR, we'll **generate our own** image-classification dataset using NumPy. The dataset consists of three geometric shapes drawn on 28×28 grayscale images:

| Label | Shape |
|-------|-------|
| 0 | Circle |
| 1 | Square |
| 2 | Triangle |

Each class will have **500 images**, with random variation in size, position, and a small amount of noise.

In [ ]:
def draw_circle(img, cx, cy, r):
    """Draw a filled circle on img."""
    yy, xx = np.ogrid[:img.shape[0], :img.shape[1]]
    mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
    img[mask] = 1.0

def draw_square(img, cx, cy, half_side):
    """Draw a filled square on img."""
    y1 = max(0, cy - half_side)
    y2 = min(img.shape[0], cy + half_side)
    x1 = max(0, cx - half_side)
    x2 = min(img.shape[1], cx + half_side)
    img[y1:y2, x1:x2] = 1.0

def draw_triangle(img, cx, cy, size):
    """Draw a filled triangle on img."""
    for row in range(img.shape[0]):
        for col in range(img.shape[1]):
            # Equilateral triangle pointing up
            top_y = cy - size
            base_y = cy + size
            if top_y <= row <= base_y:
                frac = (row - top_y) / max(1, base_y - top_y)
                half_w = frac * size
                if abs(col - cx) <= half_w:
                    img[row, col] = 1.0

def generate_shapes_dataset(n_per_class=500, img_size=28, seed=42):
    """Generate a synthetic shapes dataset."""
    rng = np.random.RandomState(seed)
    images = []
    labels = []

    for class_idx, draw_fn in enumerate([draw_circle, draw_square, draw_triangle]):
        for _ in range(n_per_class):
            img = np.zeros((img_size, img_size), dtype=np.float32)
            size = rng.randint(4, 9)
            cx = rng.randint(size + 1, img_size - size - 1)
            cy = rng.randint(size + 1, img_size - size - 1)
            draw_fn(img, cx, cy, size)
            # Add slight noise
            img += rng.randn(img_size, img_size).astype(np.float32) * 0.05
            img = np.clip(img, 0, 1)
            images.append(img)
            labels.append(class_idx)

    return np.array(images), np.array(labels)

images, labels = generate_shapes_dataset(n_per_class=500)
print(f"Dataset shape: {images.shape}")
print(f"Labels shape:  {labels.shape}")
print(f"Classes:       {np.unique(labels)} (0=circle, 1=square, 2=triangle)")

### Visualizing the Dataset

Let's look at a few samples from each class.

In [ ]:
class_names = ["Circle", "Square", "Triangle"]

fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for row, cls in enumerate(range(3)):
    idxs = np.where(labels == cls)[0][:6]
    for col, idx in enumerate(idxs):
        axes[row, col].imshow(images[idx], cmap="gray", vmin=0, vmax=1)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(class_names[cls], fontsize=12, rotation=0,
                                       labelpad=50, va="center")
plt.suptitle("Sample Images from Each Class", fontsize=14)
plt.tight_layout()
plt.show()

### PyTorch Dataset and DataLoader

We wrap our NumPy arrays in a custom `Dataset` class and split into training and test sets.

In [ ]:
class ShapesDataset(Dataset):
    """Custom Dataset for synthetic shapes."""

    def __init__(self, images, labels):
        # Add channel dimension: (N, 28, 28) -> (N, 1, 28, 28)
        self.images = torch.from_numpy(images).unsqueeze(1).float()
        self.labels = torch.from_numpy(labels).long()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

full_dataset = ShapesDataset(images, labels)
print(f"Dataset size: {len(full_dataset)}")
print(f"Image tensor shape: {full_dataset[0][0].shape}")
print(f"Label example: {full_dataset[0][1].item()}")

In [ ]:
# Split into train (80%) and test (20%)
torch.manual_seed(42)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

# Verify a batch
batch_imgs, batch_labels = next(iter(train_loader))
print(f"\nBatch image shape:  {batch_imgs.shape}")
print(f"Batch labels shape: {batch_labels.shape}")

---
# Part 6 — Training the CNN

We'll train our `ShapeCNN` with:
- **Loss function:** CrossEntropyLoss (standard for multi-class classification)
- **Optimizer:** Adam with a learning rate of 0.001
- **Epochs:** 15

We'll track both training and validation loss/accuracy each epoch.

In [ ]:
# Re-initialise model for a clean start
torch.manual_seed(42)
model = ShapeCNN(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 15
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

### Training and Validation Loop

In [ ]:
for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for batch_imgs, batch_labels in train_loader:
        batch_imgs, batch_labels = batch_imgs.to(device), batch_labels.to(device)

        optimizer.zero_grad()
        outputs = model(batch_imgs)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_imgs.size(0)
        _, predicted = outputs.max(1)
        total += batch_labels.size(0)
        correct += predicted.eq(batch_labels).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    # --- Validation ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for batch_imgs, batch_labels in test_loader:
            batch_imgs, batch_labels = batch_imgs.to(device), batch_labels.to(device)
            outputs = model(batch_imgs)
            loss = criterion(outputs, batch_labels)

            val_loss += loss.item() * batch_imgs.size(0)
            _, predicted = outputs.max(1)
            val_total += batch_labels.size(0)
            val_correct += predicted.eq(batch_labels).sum().item()

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:>2}/{num_epochs}  |  "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f}  |  "
              f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}")

print("\nTraining complete!")

### Learning Curves

Plotting loss and accuracy over epochs helps us diagnose underfitting, overfitting, or healthy convergence.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

epochs_range = range(1, num_epochs + 1)

ax1.plot(epochs_range, history["train_loss"], "o-", label="Train Loss")
ax1.plot(epochs_range, history["val_loss"], "s-", label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curve")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_acc"], "o-", label="Train Acc")
ax2.plot(epochs_range, history["val_acc"], "s-", label="Val Acc")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Curve")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

---
# Part 7 — Evaluation

Let's perform a thorough evaluation of our trained CNN.

### Collecting Predictions

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_imgs, batch_labels in test_loader:
        batch_imgs = batch_imgs.to(device)
        outputs = model(batch_imgs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

overall_acc = (all_preds == all_labels).mean()
print(f"Overall Test Accuracy: {overall_acc:.4f}")

### Per-Class Accuracy

In [ ]:
print(f"{'Class':<12} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print("-" * 40)
for cls in range(3):
    mask = all_labels == cls
    cls_correct = (all_preds[mask] == all_labels[mask]).sum()
    cls_total = mask.sum()
    cls_acc = cls_correct / cls_total if cls_total > 0 else 0
    print(f"{class_names[cls]:<12} {cls_correct:>8} {cls_total:>8} {cls_acc:>10.4f}")

### Confusion Matrix

A confusion matrix shows how often each class was predicted correctly or confused with another class.

In [ ]:
# Build confusion matrix manually (no sklearn dependency)
num_classes = 3
conf_matrix = np.zeros((num_classes, num_classes), dtype=int)
for true, pred in zip(all_labels, all_preds):
    conf_matrix[true, pred] += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(conf_matrix, cmap="Blues")
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")

# Annotate cells
for i in range(num_classes):
    for j in range(num_classes):
        color = "white" if conf_matrix[i, j] > conf_matrix.max() / 2 else "black"
        ax.text(j, i, str(conf_matrix[i, j]), ha="center", va="center",
                fontsize=14, fontweight="bold", color=color)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### Visualizing Predictions

Let's look at some test images with their predicted and actual labels. Correct predictions are shown in green, incorrect in red.

In [ ]:
# Get a batch from the test set
test_iter = iter(test_loader)
sample_imgs, sample_labels = next(test_iter)

with torch.no_grad():
    sample_outputs = model(sample_imgs.to(device))
    _, sample_preds = sample_outputs.max(1)
    sample_preds = sample_preds.cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    if i >= len(sample_imgs):
        ax.axis("off")
        continue
    ax.imshow(sample_imgs[i].squeeze(), cmap="gray", vmin=0, vmax=1)
    pred_cls = class_names[sample_preds[i]]
    true_cls = class_names[sample_labels[i]]
    is_correct = sample_preds[i] == sample_labels[i]
    color = "green" if is_correct else "red"
    ax.set_title(f"P:{pred_cls}\nT:{true_cls}", fontsize=8, color=color)
    ax.axis("off")
plt.suptitle("Predictions on Test Images (Green=Correct, Red=Wrong)", fontsize=12)
plt.tight_layout()
plt.show()

### Visualizing What the CNN Learned

Let's look at the first-layer convolutional filters — these show the low-level patterns our network learned to detect.

In [ ]:
# Extract first-layer filters
filters = model.conv1.weight.data.cpu()
print(f"Conv1 filter shape: {filters.shape}  (out_channels, in_channels, H, W)")

fig, axes = plt.subplots(2, 8, figsize=(14, 3.5))
for i, ax in enumerate(axes.flat):
    if i < filters.shape[0]:
        ax.imshow(filters[i, 0], cmap="coolwarm", vmin=-0.5, vmax=0.5)
        ax.set_title(f"Filter {i}", fontsize=8)
    ax.axis("off")
plt.suptitle("Learned Conv1 Filters (3×3)", fontsize=13)
plt.tight_layout()
plt.show()

---
# Part 8 — Transfer Learning (Conceptual Overview)

## What is Transfer Learning?

Training a CNN from scratch requires a large dataset and significant compute. **Transfer learning** sidesteps this by starting from a model already trained on a massive dataset (typically ImageNet's 1.4 million images).

### The Key Idea

A CNN's early layers learn *generic* features (edges, textures, colours) that are useful across many tasks. Only the later layers specialise to the particular dataset.

```
Pretrained model (e.g., ResNet-18 on ImageNet)
├── Early layers  →  Generic features  (FREEZE these)
├── Middle layers →  Mid-level features (optionally fine-tune)
└── Final layer   →  ImageNet classes   (REPLACE with your classes)
```

### Transfer Learning Strategies

| Strategy | What to do | When to use |
|----------|-----------|-------------|
| **Feature extraction** | Freeze all conv layers, replace & train only the classifier head | Small dataset, similar to ImageNet |
| **Fine-tuning** | Unfreeze some/all conv layers and train with a low learning rate | Medium dataset, somewhat different domain |
| **Full retraining** | Use pretrained weights as initialisation, train everything | Large dataset, very different domain |

### Popular Pretrained Models

| Model | Parameters | Top-1 Accuracy (ImageNet) | Use case |
|-------|-----------|--------------------------|----------|
| ResNet-18 | 11.7M | 69.8% | Lightweight, fast inference |
| ResNet-50 | 25.6M | 76.1% | Good balance of speed & accuracy |
| VGG-16 | 138M | 71.6% | Simple architecture, large |
| EfficientNet-B0 | 5.3M | 77.1% | Mobile / edge deployment |

### Pseudocode Example

```python
# Load a pretrained model (NOT executed here — requires download)
# import torchvision.models as models
# model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
#
# # Freeze all layers
# for param in model.parameters():
#     param.requires_grad = False
#
# # Replace the final classifier for our 3 classes
# model.fc = nn.Linear(model.fc.in_features, 3)
#
# # Only the new fc layer will be trained
# optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
```

> **Note:** We intentionally do not download pretrained models in this notebook to keep it self-contained and fast. In practice, `torchvision.models` makes it a one-liner.

---
# Part 9 — Summary and Next Steps

## Key Takeaways

1. **Images are tensors** — grayscale images have shape `(1, H, W)`, RGB images `(3, H, W)` in PyTorch.
2. **Transforms** — resizing, flipping, rotation, colour jitter, and normalisation are essential for preprocessing and augmentation.
3. **Convolutions** extract local spatial features; different kernels detect edges, blurs, and other patterns.
4. **Pooling** reduces spatial dimensions and adds translation invariance.
5. **CNN architecture** follows the pattern: Conv → ReLU → Pool → ... → Flatten → FC → Output.
6. **Training loop** uses batched gradient descent with loss tracking for both train and validation sets.
7. **Evaluation** includes per-class accuracy, confusion matrices, and visual inspection of predictions.
8. **Transfer learning** lets you leverage pretrained models to achieve strong results with limited data.

## Next Steps

| Topic | What you'll learn |
|-------|-------------------|
| **Real datasets** | MNIST, CIFAR-10, custom image folders with `torchvision.datasets` |
| **Pretrained models** | ResNet, EfficientNet, fine-tuning on your own images |
| **Data augmentation** | `transforms.Compose` pipelines for robust training |
| **Object detection** | YOLO, Faster R-CNN — locating *and* classifying objects |
| **Segmentation** | U-Net, DeepLab — pixel-level classification |
| **Deployment** | ONNX export, TorchScript, mobile inference |

Well done on completing this computer vision tutorial! 🎉

---

*This notebook is part of a series on deep learning with PyTorch. Previous: Deep Learning Basics → **Computer Vision** → Next: NLP & Sequence Models*